# Task A Run 18 -- does R-Drop help? The short test

R-Drop is in Task B's winning recipe but **has never been measured on Task A**. Run 5 would
answer it, but it is a 6.5-hour full-data pair with five seeds each and no local score. This
is the cheapest design that actually answers the question.

| | |
|---|---|
| split | the fixed 15% holdout, 960 rows |
| encoder | stock `google/muril-base-cased` |
| recipe | 2 reinitialized layers, 6 epochs, effective batch 16, `--select last`, seed 42 |
| the one difference | `--rdrop 0` against `--rdrop 0.5` |
| runtime | **about 80 minutes** |

## Why the holdout rather than five folds

Five folds would cost 320 minutes for the pair. The holdout costs 80. Its noise is about
1.3 points for a single score, which would normally be too coarse for an effect this size —
but both arms are scored on **the same 960 rows**, so the comparison is paired and the
noise on the *difference* is much smaller than the noise on either score. The last cell
bootstraps that paired difference and reports an interval, so the answer comes with its own
uncertainty rather than as a bare number.

## What is deliberately not in it

No TAPT, no 10 epochs, no transductive rows. Each would make the test more representative
of the final recipe and each would make it slower, and the question here is whether R-Drop
helps at all. If it does, the follow-up is whether it still helps on top of TAPT.

Set **Accelerator** to `GPU T4 x2` or `GPU P100` and **Internet** on, then
**Save Version -> Save & Run All**.

In [ ]:
import json, os, pathlib, shutil, subprocess, sys, zipfile

WORK = "/kaggle/working/hastika"
if os.path.isdir(WORK + "/.git"):
    subprocess.run(["git", "-C", WORK, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "-q", "-b", "task-b", "--depth", "1",
                    "https://github.com/robinpnalex/Hastika-ICON2026.git", WORK], check=True)
os.chdir(WORK)
os.environ["PYTHONPATH"] = os.path.join(WORK, "src")
sys.path.insert(0, os.path.join(WORK, "src"))
pathlib.Path("artifacts/logs").mkdir(parents=True, exist_ok=True)
print("repo:", os.getcwd())
subprocess.run(["git", "log", "-1", "--oneline"], check=True)
subprocess.run('pip install -q emoji ftfy sentencepiece protobuf "transformers>=4.45,<6"',
               shell=True, check=True)

import numpy as np
import pandas as pd
import torch
from hastika.common.preprocessing import dedupe_index

assert torch.cuda.is_available(), "no GPU -- select a CUDA-enabled runtime"
print("gpu:", torch.cuda.get_device_name(0))
train = pd.read_csv("data/raw/binary_train.csv")
keep = dedupe_index(train["Comment"].tolist(), train["Label"].tolist(), "task A")
print(f"raw labelled rows: {len(train)}; deduplicated rows used for fitting: {len(keep)}")
assert len(keep) == 6401, len(keep)

def run(cmd, log=None):
    print("$", " ".join(cmd), flush=True)
    fh = open(log, "w") if log else None
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1)
    for line in p.stdout:
        sys.stdout.write(line)
        if fh:
            fh.write(line)
    p.wait()
    if fh:
        fh.close()
    if p.returncode:
        raise RuntimeError(f"exit {p.returncode}: {cmd}")


## 1. Two arms, one flag apart

`--folds 0` uses the repository's fixed stratified 15% split with seed 42, so both arms
hold out exactly the same rows.

In [ ]:
COMMON = ["--model", "google/muril-base-cased", "--folds", "0", "--epochs", "6",
          "--bs", "8", "--grad-accum", "2", "--eval-bs", "32", "--select", "last",
          "--reinit-layers", "2", "--seeds", "42"]
ARMS = [("a_rdrop_off", ["--rdrop", "0"]), ("a_rdrop_on", ["--rdrop", "0.5"])]
for tag, extra in ARMS:
    run([sys.executable, "-u", "-m", "hastika.models.muril", "--tag", tag, *COMMON, *extra],
        log=f"artifacts/logs/{tag}.log")

## 2. Score both arms on the same rows

`holdout_probs.npy` is a full-length matrix with zeros on the rows the arm trained on, so
the held-out rows are the ones with a non-zero row, and both arms share them.

In [ ]:
from sklearn.metrics import f1_score
from hastika.common.preprocessing import clean

df = train.iloc[keep].reset_index(drop=True)
y = (df["Label"] == "Hate").astype(int).values
P = {tag: np.load(pathlib.Path("artifacts/runs") / tag / "holdout_probs.npy")
     for tag, _ in ARMS}
masks = {t: p.sum(1) > 0 for t, p in P.items()}
assert (masks["a_rdrop_off"] == masks["a_rdrop_on"]).all(), "arms scored different rows"
m = masks["a_rdrop_off"]
yh = y[m]
pred = {t: P[t][m].argmax(1) for t in P}
for t in pred:
    print(f"  {t:14s} holdout macro-F1 {f1_score(yh, pred[t], average='macro'):.4f}")
delta = f1_score(yh, pred["a_rdrop_on"], average="macro") - f1_score(yh, pred["a_rdrop_off"], average="macro")
print(f"\nR-Drop effect: {delta:+.4f} on {int(m.sum())} rows")
print(f"the two arms disagree on {int((pred['a_rdrop_on'] != pred['a_rdrop_off']).sum())} rows")

## 3. Is the difference bigger than the noise?

Both arms are scored on the same rows, so the difference can be bootstrapped directly:
resample the held-out rows, rescore both arms on that resample, take the difference. The
interval below is the honest readout, not the point estimate above.

In [ ]:
rng = np.random.default_rng(0)
idx = np.arange(len(yh))
diffs = []
for _ in range(2000):
    b = rng.integers(0, len(idx), len(idx))
    diffs.append(f1_score(yh[b], pred["a_rdrop_on"][b], average="macro")
                 - f1_score(yh[b], pred["a_rdrop_off"][b], average="macro"))
diffs = np.array(diffs)
lo, hi = np.percentile(diffs, [2.5, 97.5])
print(f"paired bootstrap of the R-Drop effect: {diffs.mean():+.4f}  95% interval [{lo:+.4f}, {hi:+.4f}]")
print(f"P(R-Drop helps) = {(diffs > 0).mean():.3f}")
if lo > 0:
    print("\nR-Drop helps: the interval excludes zero. Add it to the Task A recipe.")
elif hi < 0:
    print("\nR-Drop hurts: the interval excludes zero. Leave it out.")
else:
    print("\nUnresolved: the interval spans zero. Treat R-Drop as untested on Task A"
          " and leave it out, since the recipe is simpler without it.")

## 4. Preserve outputs

In [ ]:
OUT = pathlib.Path("/kaggle/working/task_a_rdrop_outputs")
OUT.mkdir(parents=True, exist_ok=True)
for tag, _ in ARMS:
    shutil.copy2(pathlib.Path("artifacts/runs") / tag / "holdout_probs.npy",
                 OUT / f"{tag}_holdout_probs.npy")
    shutil.copy2(f"artifacts/logs/{tag}.log", OUT / f"{tag}.log")
json.dump({"delta": float(delta), "ci_low": float(lo), "ci_high": float(hi),
           "p_helps": float((diffs > 0).mean())}, open(OUT / "rdrop_result.json", "w"), indent=2)
print(sorted(x.name for x in OUT.iterdir()))

## 5. What to do with the result

Record both arm scores and the interval in `docs/EXPERIMENTS.md`, including a null result.
R-Drop is currently in Task B's winning recipe on the strength of a 1.1-point CodaBench gap
that is a third of that set's standard deviation, so a clean null here is informative for
both tasks.

If R-Drop helps, the follow-up question is whether it still helps on top of TAPT, which
this test deliberately does not answer.